# Project FORESIGHT — Notebook 04: Inventory Risk & Decisioning

Converts the production demand forecast (**seasonal-naive baseline** — the HistGradientBoosting
model did not beat it on backtest, see `reports/forecast_summary.txt`, and is not used here)
into a stockout / overstock risk score and a recommended action per SKU.

## Business Logic (from the FORESIGHT engagement brief)

- **Stockout risk**: forecast demand over the replenishment lead time is compared against
  on-hand + on-order stock. If projected stock after the lead time falls below the safety-stock
  level, the SKU is flagged.
- **Overstock risk**: current on-hand stock is compared against forecast demand over the
  6-week forward window. If stock held is more than `OVERSTOCK_MULTIPLIER` × forward demand,
  the SKU is flagged.
- Every threshold is a **documented business-rule assumption**, not a machine-learning result,
  and lives as a named constant in `src/risk.py`.

| stockout_risk | overstock_risk | Action |
|---|---|---|
| False | False | HEALTHY |
| True | False | REORDER NOW |
| False | True | MARKDOWN / CLEAR |
| True | True | WATCH / VOLATILE |

Rupee figures are **estimated exposure**, not confirmed financial losses.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import risk  # src/risk.py

risk_table, latest_snapshot_date = risk.build_risk_table()
print(f"Latest inventory snapshot used: {latest_snapshot_date.date()}")
print(f"Overstock multiplier (business rule): {risk.OVERSTOCK_MULTIPLIER}")
print(f"Rows: {len(risk_table)}  (should equal the 50 modeled SKUs)")
assert risk_table['SKU'].is_unique, "Duplicate SKU rows in risk table"
assert len(risk_table) == 50, "Expected exactly one risk row per modeled SKU"
risk_table.head(3)

Latest inventory snapshot used: 2025-12-01
Overstock multiplier (business rule): 2.0
Rows: 50  (should equal the 50 modeled SKUs)


,SKU,Product_Name,Category,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,forward_demand_6w,lead_time_demand,...,stockout_risk,overstock_risk,risk_action,sales_at_risk_rs,capital_locked_rs,total_rupee_impact_rs,stockout_severity,overstock_severity,rupee_severity,priority_score
0,SKU012,Product 012,Home Decor,47,60,12,16,58,1135,374,...,True,False,REORDER NOW,2484561.71,0.00,2484561.71,16.647059,0.000000,1.000000,17.647059
1,SKU025,Product 025,Storage,1124,61,14,121,582,115,33,...,False,True,MARKDOWN / CLEAR,0.00,1191773.52,1191773.52,0.000000,7.706897,0.479672,8.186568
2,SKU031,Product 031,Furniture,93,40,13,28,113,863,277,...,True,False,REORDER NOW,1499139.96,0.00,1499139.96,5.931034,0.000000,0.603382,6.534417


## 1. Inventory Risk Overview

In [2]:
action_counts = risk_table["risk_action"].value_counts().reindex(
    ["REORDER NOW", "WATCH / VOLATILE", "MARKDOWN / CLEAR", "HEALTHY"]).fillna(0).astype(int)
print(action_counts.to_string())
print()
print(f"Total estimated sales at risk (stockouts): Rs {risk_table['sales_at_risk_rs'].sum():,.0f}")
print(f"Total estimated capital locked (overstock): Rs {risk_table['capital_locked_rs'].sum():,.0f}")
print(f"Total estimated rupee impact: Rs {risk_table['total_rupee_impact_rs'].sum():,.0f}")

risk_action
REORDER NOW          8
WATCH / VOLATILE     0
MARKDOWN / CLEAR     2
HEALTHY             40

Total estimated sales at risk (stockouts): Rs 6,524,173
Total estimated capital locked (overstock): Rs 2,111,118
Total estimated rupee impact: Rs 8,635,291


## 2. SKUs by Action

In [3]:
fig = px.bar(action_counts, x=action_counts.index, y=action_counts.values,
             title="Number of SKUs by Recommended Action",
             labels={"x": "Action", "y": "Number of SKUs"},
             color=action_counts.index,
             color_discrete_map={"REORDER NOW": "firebrick", "WATCH / VOLATILE": "goldenrod",
                                  "MARKDOWN / CLEAR": "steelblue", "HEALTHY": "seagreen"},
             template="plotly_white")
fig.update_layout(showlegend=False)
fig.show()

## 3. Stockout Risk Distribution

In [4]:
fig = px.histogram(risk_table, x="projected_stock_after_lead", color="stockout_risk",
                    title="Projected Stock After Lead Time (vs Safety Stock)",
                    labels={"projected_stock_after_lead": "Projected stock after lead time (units)"},
                    template="plotly_white", nbins=20)
fig.show()

n_stockout = int(risk_table["stockout_risk"].sum())
print(f"{n_stockout} of {len(risk_table)} SKUs are projected to fall below safety stock "
      f"within their replenishment lead time.")

8 of 50 SKUs are projected to fall below safety stock within their replenishment lead time.


## 4. Overstock Risk Distribution

In [5]:
fig = px.histogram(risk_table, x="inventory_coverage_weeks", color="overstock_risk",
                    title="Inventory Coverage (Weeks of Stock on Hand vs Forecast Demand)",
                    labels={"inventory_coverage_weeks": "Weeks of stock on hand"},
                    template="plotly_white", nbins=20)
fig.show()

n_overstock = int(risk_table["overstock_risk"].sum())
print(f"{n_overstock} of {len(risk_table)} SKUs hold more than "
      f"{risk.OVERSTOCK_MULTIPLIER}x their 6-week forecast demand in current stock.")

2 of 50 SKUs hold more than 2.0x their 6-week forecast demand in current stock.


## 5 & 6. Total Estimated Sales-at-Risk and Capital Locked

In [6]:
impact_summary = pd.DataFrame({
    "Impact Type": ["Sales at Risk (Stockout)", "Capital Locked (Overstock)"],
    "Rupees": [risk_table["sales_at_risk_rs"].sum(), risk_table["capital_locked_rs"].sum()],
})
fig = px.bar(impact_summary, x="Impact Type", y="Rupees",
             title="Total Estimated Rupee Exposure by Risk Type",
             labels={"Rupees": "Estimated Rupee Exposure (Rs)"}, template="plotly_white")
fig.show()

## 7. Top REORDER NOW SKUs

In [7]:
top_reorder = risk_table[risk_table["risk_action"] == "REORDER NOW"].sort_values(
    "priority_score", ascending=False).head(10)
top_reorder[["SKU", "Product_Name", "Category", "Current_Stock", "estimated_units_short",
             "sales_at_risk_rs", "priority_score"]]

,SKU,Product_Name,Category,Current_Stock,estimated_units_short,sales_at_risk_rs,priority_score
0,SKU012,Product 012,Home Decor,47,283,2484561.71,17.647059
2,SKU031,Product 031,Furniture,93,172,1499139.96,6.534417
4,SKU040,Product 040,Storage,67,167,409243.52,4.678228
5,SKU010,Product 010,Storage,83,100,66346.00,4.374529
6,SKU017,Product 017,Home Decor,95,190,1610690.80,2.391399
7,SKU018,Product 018,Kitchen,235,65,362401.65,1.116011
8,SKU023,Product 023,Kitchen,130,38,53836.12,0.926430
9,SKU034,Product 034,Lighting,34,25,37953.50,0.729561


## 8. Top MARKDOWN / CLEAR SKUs

In [8]:
top_markdown = risk_table[risk_table["risk_action"] == "MARKDOWN / CLEAR"].sort_values(
    "priority_score", ascending=False).head(10)
top_markdown[["SKU", "Product_Name", "Category", "Current_Stock", "excess_inventory",
              "capital_locked_rs", "priority_score"]]

,SKU,Product_Name,Category,Current_Stock,excess_inventory,capital_locked_rs,priority_score
1,SKU025,Product 025,Storage,1124,894.0,1191773.52,8.186568
3,SKU011,Product 011,Furniture,753,535.0,919344.00,5.233659


## 9. Risk Matrix

Every SKU placed on a stockout-severity vs overstock-severity grid, colored by action and
sized by rupee impact — the same decisioning view described in the engagement brief.

In [9]:
fig = px.scatter(
    risk_table, x="overstock_severity", y="stockout_severity",
    color="risk_action", size="total_rupee_impact_rs", hover_name="Product_Name",
    hover_data=["SKU", "Category", "total_rupee_impact_rs"],
    color_discrete_map={"REORDER NOW": "firebrick", "WATCH / VOLATILE": "goldenrod",
                         "MARKDOWN / CLEAR": "steelblue", "HEALTHY": "seagreen"},
    title="Stockout vs Overstock Risk Matrix (bubble size = estimated rupee impact)",
    labels={"overstock_severity": "Overstock severity →", "stockout_severity": "Stockout severity ↑"},
    template="plotly_white", size_max=40)
fig.update_layout(height=550)
fig.show()

## 10. Final Decision Table

Sorted by `priority_score` (stockout severity + overstock severity + relative rupee impact),
so the highest-priority SKUs surface first for the dashboard.

In [10]:
decision_table = risk_table[[
    "SKU", "Product_Name", "Category", "risk_action", "Current_Stock", "estimated_units_short",
    "excess_inventory", "sales_at_risk_rs", "capital_locked_rs", "total_rupee_impact_rs",
    "priority_score",
]].copy()
decision_table

,SKU,Product_Name,Category,risk_action,Current_Stock,estimated_units_short,excess_inventory,sales_at_risk_rs,capital_locked_rs,total_rupee_impact_rs,priority_score
0,SKU012,Product 012,Home Decor,REORDER NOW,47,283,0.0,2484561.71,0.00,2484561.71,17.647059
1,SKU025,Product 025,Storage,MARKDOWN / CLEAR,1124,0,894.0,0.00,1191773.52,1191773.52,8.186568
2,SKU031,Product 031,Furniture,REORDER NOW,93,172,0.0,1499139.96,0.00,1499139.96,6.534417
3,SKU011,Product 011,Furniture,MARKDOWN / CLEAR,753,0,535.0,0.00,919344.00,919344.00,5.233659
4,SKU040,Product 040,Storage,REORDER NOW,67,167,0.0,409243.52,0.00,409243.52,4.678228
5,SKU010,Product 010,Storage,REORDER NOW,83,100,0.0,66346.00,0.00,66346.00,4.374529
6,SKU017,Product 017,Home Decor,REORDER NOW,95,190,0.0,1610690.80,0.00,1610690.80,2.391399
7,SKU018,Product 018,Kitchen,REORDER NOW,235,65,0.0,362401.65,0.00,362401.65,1.116011
8,SKU023,Product 023,Kitchen,REORDER NOW,130,38,0.0,53836.12,0.00,53836.12,0.926430
9,SKU034,Product 034,Lighting,REORDER NOW,34,25,0.0,37953.50,0.00,37953.50,0.729561


## Data-Quality & Methodology Notes

- Exactly one inventory record per SKU was used (the latest available monthly snapshot) —
  no future inventory information and no repeated daily values were treated as independent
  measurements.
- The forecast used is the **production seasonal-naive baseline**; the HistGradientBoosting
  model is not used for any decision here since it did not beat the baseline on backtest.
- `OVERSTOCK_MULTIPLIER` is a business-rule assumption set at the top of `src/risk.py` — the
  ops team can tune it without touching any modeling code.
- Rupee figures are **estimated exposure**, not confirmed financial outcomes.
- Zero-demand SKUs are handled explicitly: any positive stock with zero forward forecast is
  flagged overstock; inventory-coverage weeks is left undefined (not divided by zero) when
  forecast demand is zero.

## Save Outputs

In [11]:
risk.save_outputs(risk_table, latest_snapshot_date)
print("Saved:")
print(" - data/processed/risk_output.csv")
print(" - reports/risk_metrics.csv")
print(" - reports/risk_summary.txt")

Saved:
 - data/processed/risk_output.csv
 - reports/risk_metrics.csv
 - reports/risk_summary.txt
